# 1. Data Retrieval: download, clean and chunk the annual reports

**Goal:** turn 5 company annual reports (10-K filings) into small, labeled text chunks the chatbot can search.

```
SEC website ──download──► raw HTML ──clean──► plain text ──chunk──► data/chunks.csv
```

| Company | Ticker |
|---|---|
| Apple | AAPL |
| Microsoft | MSFT |
| NVIDIA | NVDA |
| Tesla | TSLA |
| Amazon | AMZN |

**Where the data comes from:** every US-listed company files its annual report (**10-K**) with the SEC (US Securities and Exchange Commission). The reports are free and public.

## Setup

The SEC only accepts downloads from programs that say who they are: your **name and email** in the `User-Agent` header.
They're stored in `.env` as `SEC_USER_AGENT`, never written in the code.

In [1]:
from pathlib import Path

# Project root = the folder that contains requirements.txt (works no matter where the notebook runs from)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Project folder:", ROOT)

Project folder: /Users/os_bhosale/Desktop/Projects/Financial-Filings-Rag


In [2]:
import os
import re
import json
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(ROOT / ".env")
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}
print("Name and email found:", HEADERS["User-Agent"] is not None)

TICKERS = ["AAPL", "MSFT", "NVDA", "TSLA", "AMZN"]
RAW_DIR, TEXT_DIR = ROOT / "data" / "raw", ROOT / "data" / "text"
RAW_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Name and email found: True


## Step 1: Find each company's ID number (CIK)

We know companies by their ticker (`AAPL`), but the SEC organizes reports by its own ID, the **CIK**.
The SEC publishes one file listing every company's ticker and CIK; we look up our 5.

In [3]:
response = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS)
response.raise_for_status()

ticker_to_cik = {company["ticker"]: company["cik_str"] for company in response.json().values()}
companies = {ticker: ticker_to_cik[ticker] for ticker in TICKERS}
companies

{'AAPL': 320193,
 'MSFT': 789019,
 'NVDA': 1045810,
 'TSLA': 1318605,
 'AMZN': 1018724}

## Step 2: Find each company's latest annual report

Each company's filing list is sorted **newest first**. We go down it and stop at the first `10-K`.
The report's address is built from the CIK, the filing's ID (accession number) and the document name.

`time.sleep(0.2)` keeps us under the SEC's limit of 10 requests per second.

In [4]:
def find_latest_10k(cik):
    url = f"https://data.sec.gov/submissions/CIK{str(cik).zfill(10)}.json"
    recent = requests.get(url, headers=HEADERS).json()["filings"]["recent"]
    for i in range(len(recent["form"])):
        if recent["form"][i] == "10-K":
            accession = recent["accessionNumber"][i].replace("-", "")
            return {
                "filing_date": recent["filingDate"][i],
                "url": f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{recent['primaryDocument'][i]}",
            }
    return None


filings = {}
for ticker, cik in companies.items():
    filings[ticker] = find_latest_10k(cik)
    print(ticker, filings[ticker]["filing_date"], filings[ticker]["url"])
    time.sleep(0.2)

AAPL 2026-11-03 https://www.sec.gov/Archives/edgar/data/320193/000032019326000026/aapl-20260926.htm


MSFT 2026-07-29 https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm


NVDA 2026-02-25 https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm


TSLA 2026-01-29 https://www.sec.gov/Archives/edgar/data/1318605/000162828026003952/tsla-20251231.htm


AMZN 2026-02-06 https://www.sec.gov/Archives/edgar/data/1018724/000101872426000004/amzn-20251231.htm


The end of each filename is the date the company's **fiscal year ended**, which differs by company
(e.g. Apple's year ends in September, Microsoft's in June, Tesla's and Amazon's in December).

## Step 3: Download the reports (once)

Files already on disk are skipped, so re-running never downloads twice.
Raw bytes (`"wb"`) are saved exactly as received, which avoids encoding problems.

In [5]:
for ticker, info in filings.items():
    path = RAW_DIR / f"{ticker}_10K.htm"
    if path.exists():
        print(ticker, "already downloaded")
        continue
    response = requests.get(info["url"], headers=HEADERS)
    response.raise_for_status()
    path.write_bytes(response.content)
    print(ticker, f"saved ({len(response.content) / 1e6:.1f} MB)")
    time.sleep(0.2)

with open(RAW_DIR / "filings.json", "w") as f:
    json.dump(filings, f, indent=2)

AAPL already downloaded
MSFT already downloaded
NVDA already downloaded
TSLA already downloaded
AMZN already downloaded


## Step 4: Turn the HTML into clean text

Two problems with a plain `get_text()`:
1. **Hidden data:** reports embed invisible machine-readable data (XBRL), which shows up as junk like `P1Y`. We delete hidden parts.
2. **Broken tables:** every table cell lands on its own line, separating labels from numbers.
   We rebuild each row as one line: `Total net sales | 478,834 | 15 | 416,161 | 6 | 391,035`.

Then we tidy spaces and empty lines.

In [6]:
def html_to_text(path):
    soup = BeautifulSoup(path.read_bytes(), "html.parser")

    # 1. Remove hidden content
    for tag in soup.find_all("ix:header"):
        tag.decompose()
    for tag in soup.find_all(style=re.compile(r"display:\s*none")):
        tag.decompose()

    # 2. Rebuild every table row as one line
    for table in soup.find_all("table"):
        rows = []
        for tr in table.find_all("tr"):
            cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
            cells = [c for c in cells if c not in ("", "$", "%", ")")]
            if cells:
                rows.append(" | ".join(cells))
        table.replace_with("\n" + "\n".join(rows) + "\n")

    # 3. Tidy spaces and empty lines
    text = soup.get_text("\n").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


texts = {}
for ticker in TICKERS:
    texts[ticker] = html_to_text(RAW_DIR / f"{ticker}_10K.htm")
    (TEXT_DIR / f"{ticker}.txt").write_text(texts[ticker], encoding="utf-8")
    print(ticker, f"{len(texts[ticker]):,} characters")

AAPL 235,879 characters


MSFT 327,633 characters
NVDA 343,235 characters


TSLA 399,429 characters
AMZN 286,828 characters


In [7]:
i = texts["AAPL"].find("Total net sales")
print(texts["AAPL"][i - 200 : i + 60])


Europe | 127,604 | 15 | 111,032 | 10 | 101,328
Greater China | 82,378 | 28 | 64,377 | (4) | 66,952
Japan | 30,908 | 8 | 28,703 | 15 | 25,052
Rest of Asia Pacific | 39,861 | 18 | 33,696 | 10 | 30,658
Total net sales | 478,834 | 15 | 416,161 | 6 | 391,035

Amer


Each table row now stays on one line, with its label next to its numbers (in millions of US dollars; `(4)` means -4%).

## Step 5: Split each report into its sections

Every 10-K has the same numbered sections by law: *Item 1. Business*, *Item 1A. Risk Factors*,
*Item 7. Management's Discussion and Analysis*, *Item 8. Financial Statements*…

Each section becomes a LangChain `Document` labeled with its company and section, so every chunk can be **cited**.

- **Table of contents:** every report lists its sections twice. The real report starts at the **second** "Item 1".
- **Standard names:** headings are formatted differently by each company (tables, capitals, split lines),
  so we use the standard section names instead of copying the heading text.
- Note: NVIDIA places its financial statements under **Item 15** instead of Item 8.

In [8]:
SECTION_NAMES = {
    "1": "Business", "1A": "Risk Factors", "1B": "Unresolved Staff Comments", "1C": "Cybersecurity",
    "2": "Properties", "3": "Legal Proceedings", "4": "Mine Safety Disclosures",
    "5": "Market for Common Equity", "6": "[Reserved]",
    "7": "Management's Discussion and Analysis", "7A": "Market Risk Disclosures",
    "8": "Financial Statements", "9": "Changes in Accountants", "9A": "Controls and Procedures",
    "9B": "Other Information", "9C": "Foreign Jurisdiction Disclosures",
    "10": "Directors and Corporate Governance", "11": "Executive Compensation",
    "12": "Security Ownership", "13": "Related Transactions", "14": "Accountant Fees",
    "15": "Exhibits and Schedules", "16": "Form 10-K Summary",
}
HEADING = re.compile(r"^item\s+(\d{1,2}[a-c]?)(?![0-9a-z])", re.IGNORECASE)


def split_into_sections(text, ticker):
    lines = text.split("\n")

    # Find every short line that starts with "Item <number>"
    headings = []
    for i, line in enumerate(lines):
        match = HEADING.match(line.strip())
        if match and len(line) < 200 and match.group(1).upper() in SECTION_NAMES:
            headings.append((i, match.group(1).upper()))

    # Skip the table of contents: the real report starts at the second "Item 1"
    item1_lines = [i for i, item in headings if item == "1"]
    body_start = item1_lines[1] if len(item1_lines) > 1 else (item1_lines[0] if item1_lines else len(lines))
    section_starts = {i: item for i, item in headings if i >= body_start}

    documents, buffer, section = [], [], "Cover page"

    def save():
        content = "\n".join(buffer).strip()
        if content:
            documents.append(Document(page_content=content, metadata={"ticker": ticker, "section": section}))

    for i, line in enumerate(lines):
        if i in section_starts:
            new_section = f"Item {section_starts[i]}. {SECTION_NAMES[section_starts[i]]}"
            if new_section != section:          # repeated page headers don't start a new section
                save()
                buffer, section = [], new_section
        buffer.append(line)
    save()
    return documents


sections = []
for ticker in TICKERS:
    sections += split_into_sections(texts[ticker], ticker)
print(len(sections), "sections")

140 sections


## Step 6: Cut the sections into chunks (LangChain)

`RecursiveCharacterTextSplitter` cuts each section into chunks of up to **1,500 characters** (~230 words),
preferring to cut between paragraphs, then between lines (table rows), then between words.

- **Overlap of 200 characters:** a sentence cut at a boundary still appears complete in one chunk.
- `split_documents` copies each section's labels onto all of its chunks.

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
chunk_docs = splitter.split_documents(sections)

chunks = pd.DataFrame([
    {"ticker": d.metadata["ticker"], "section": d.metadata["section"], "text": d.page_content}
    for d in chunk_docs
])
chunks["chunk_id"] = chunks["ticker"] + "_" + chunks.groupby("ticker").cumcount().astype(str)
chunks.to_csv(ROOT / "data" / "chunks.csv", index=False)

print(len(chunks), "chunks saved to data/chunks.csv")
chunks.groupby("ticker").size()

1633 chunks saved to data/chunks.csv


ticker
AAPL    244
AMZN    286
MSFT    364
NVDA    324
TSLA    415
dtype: int64

In [10]:
# Chunks per section for one company
chunks[chunks["ticker"] == "AMZN"].groupby("section", sort=False).size()

section
Cover page                                        6
Item 1. Business                                 14
Item 1A. Risk Factors                            57
Item 1B. Unresolved Staff Comments                1
Item 1C. Cybersecurity                            5
Item 2. Properties                                1
Item 3. Legal Proceedings                         1
Item 4. Mine Safety Disclosures                   1
Item 5. Market for Common Equity                  1
Item 6. [Reserved]                                1
Item 7. Management's Discussion and Analysis     49
Item 7A. Market Risk Disclosures                  7
Item 8. Financial Statements                    112
Item 9. Changes in Accountants                    1
Item 9A. Controls and Procedures                  8
Item 9B. Other Information                        3
Item 9C. Foreign Jurisdiction Disclosures         1
Item 10. Directors and Corporate Governance       1
Item 11. Executive Compensation                   1
Item

In [11]:
example = chunks[chunks["text"].str.contains("Total net sales")].iloc[0]
print(example["chunk_id"], "|", example["section"])
print(example["text"][:500])

AAPL_137 | Item 7. Management's Discussion and Analysis
2026 | Change | 2025 | Change | 2024
Americas | 198,083 | 11 | 178,353 | 7 | 167,045
Europe | 127,604 | 15 | 111,032 | 10 | 101,328
Greater China | 82,378 | 28 | 64,377 | (4) | 66,952
Japan | 30,908 | 8 | 28,703 | 15 | 25,052
Rest of Asia Pacific | 39,861 | 18 | 33,696 | 10 | 30,658
Total net sales | 478,834 | 15 | 416,161 | 6 | 391,035


**Output:** `data/chunks.csv`, one row per chunk with its company, section, text and a unique `chunk_id` (e.g. `AAPL_139`).

Next: `2_embedding/embedding.ipynb` makes the chunks searchable by meaning.